# Build historical customer-product features

## Goal

Add point-in-time-correct history features to each row in `cleaned_purchases.csv`. Every feature uses only rows that occurred strictly before the row's `purchase_date`.

This notebook produces an intermediate historical feature table. It does **not** yet produce the final CatBoost ranking table because negative candidate products and labels still need to be added.

## 1. Setup

In [ ]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
input_path = project_root / "data" / "interim" / "cleaned_purchases.csv"
output_path = (
    project_root
    / "data"
    / "processed"
    / "historical_purchase_features.csv"
)

## 2. Load and validate cleaned purchases

Dates are parsed explicitly, and identifier columns are loaded as strings. The input must already contain one row per customer, date, and product.

In [ ]:
identifier_dtypes = {
    "customer_id": "string",
    "product_id": "string",
    "product_category": "string",
    "business_line": "string",
    "promotion_name": "string",
}

df = pd.read_csv(
    input_path,
    dtype=identifier_dtypes,
    parse_dates=["purchase_date"],
)

required_columns = {
    "customer_id",
    "purchase_date",
    "product_id",
    "paid_quantity",
    "gift_quantity",
    "received_quantity",
}
missing_columns = sorted(required_columns - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

grouping_columns = ["customer_id", "product_id"]
event_key_columns = ["customer_id", "purchase_date", "product_id"]

df = (
    df
    .sort_values(grouping_columns + ["purchase_date"])
    .reset_index(drop=True)
)

duplicate_count = int(df.duplicated(event_key_columns).sum())
assert duplicate_count == 0, f"Found {duplicate_count} duplicate event keys."

print(f"Loaded {len(df):,} cleaned purchase rows.")
df.head(10)

## 3. Count events strictly before each row

The temporary indicator columns mark whether the current row contains a paid purchase, gift, or receipt. A cumulative sum counts events up to the current row; subtracting the current indicator leaves only earlier events.

In [ ]:
df["_is_paid_purchase"] = df["paid_quantity"].gt(0).astype("int64")
df["_is_gift"] = df["gift_quantity"].gt(0).astype("int64")
df["_is_receipt"] = df["received_quantity"].gt(0).astype("int64")

feature_indicator_pairs = {
    "previous_paid_purchase_count": "_is_paid_purchase",
    "previous_gift_count": "_is_gift",
    "previous_receipt_count": "_is_receipt",
}

for feature_column, indicator_column in feature_indicator_pairs.items():
    cumulative_count = (
        df
        .groupby(grouping_columns, sort=False)[indicator_column]
        .cumsum()
    )
    df[feature_column] = cumulative_count - df[indicator_column]

df["previous_paid_quantity"] = (
    df.groupby(grouping_columns, sort=False)["paid_quantity"].cumsum()
    - df["paid_quantity"]
)
df["previous_gift_quantity"] = (
    df.groupby(grouping_columns, sort=False)["gift_quantity"].cumsum()
    - df["gift_quantity"]
)
df["previous_received_quantity"] = (
    df.groupby(grouping_columns, sort=False)["received_quantity"].cumsum()
    - df["received_quantity"]
)

df["average_previous_received_quantity"] = (
    df["previous_received_quantity"]
    / df["previous_receipt_count"].replace(0, pd.NA)
)

df[[
    "customer_id",
    "product_id",
    "purchase_date",
    "previous_paid_purchase_count",
    "previous_gift_count",
    "previous_receipt_count",
]].head(10)

## 4. Add previous dates and recency features

`shift(1)` moves the previous row's value into the current row. Paid and gift dates are forward-filled within each customer-product history before shifting, so a gift does not erase the last paid-purchase date and a paid purchase does not erase the last gift date.

In [ ]:
df["previous_received_date"] = (
    df
    .groupby(grouping_columns, sort=False)["purchase_date"]
    .shift(1)
)
df["last_received_quantity"] = (
    df
    .groupby(grouping_columns, sort=False)["received_quantity"]
    .shift(1)
)

df["_paid_purchase_date"] = df["purchase_date"].where(
    df["paid_quantity"].gt(0)
)
df["_gift_date"] = df["purchase_date"].where(
    df["gift_quantity"].gt(0)
)

last_paid_date_so_far = (
    df
    .groupby(grouping_columns, sort=False)["_paid_purchase_date"]
    .ffill()
)
last_gift_date_so_far = (
    df
    .groupby(grouping_columns, sort=False)["_gift_date"]
    .ffill()
)

df["previous_paid_purchase_date"] = (
    last_paid_date_so_far
    .groupby([df["customer_id"], df["product_id"]], sort=False)
    .shift(1)
)
df["previous_gift_date"] = (
    last_gift_date_so_far
    .groupby([df["customer_id"], df["product_id"]], sort=False)
    .shift(1)
)

df["days_since_last_received"] = (
    df["purchase_date"] - df["previous_received_date"]
).dt.days
df["days_since_last_paid_purchase"] = (
    df["purchase_date"] - df["previous_paid_purchase_date"]
).dt.days
df["days_since_last_gift"] = (
    df["purchase_date"] - df["previous_gift_date"]
).dt.days

df["month"] = df["purchase_date"].dt.month
df["weekday"] = df["purchase_date"].dt.dayofweek

df[[
    "customer_id",
    "product_id",
    "purchase_date",
    "previous_received_date",
    "days_since_last_received",
    "previous_paid_purchase_date",
    "days_since_last_paid_purchase",
]].head(10)

## 5. Validate and save

The validation checks that history counts are non-negative and that every previous date is strictly earlier than the current row's date. Temporary calculation columns are removed before saving.

In [ ]:
history_count_columns = list(feature_indicator_pairs)
negative_history_count = int(df[history_count_columns].lt(0).sum().sum())

previous_date_columns = [
    "previous_received_date",
    "previous_paid_purchase_date",
    "previous_gift_date",
]
invalid_previous_dates = sum(
    int(
        (
            df[date_column].notna()
            & df[date_column].ge(df["purchase_date"])
        ).sum()
    )
    for date_column in previous_date_columns
)

first_history_rows = df.groupby(grouping_columns, sort=False).head(1)
nonzero_first_history_count = int(
    first_history_rows[history_count_columns].ne(0).sum().sum()
)

assert negative_history_count == 0
assert invalid_previous_dates == 0
assert nonzero_first_history_count == 0

temporary_columns = [
    "_is_paid_purchase",
    "_is_gift",
    "_is_receipt",
    "_paid_purchase_date",
    "_gift_date",
]
historical_features = df.drop(columns=temporary_columns)

output_path.parent.mkdir(parents=True, exist_ok=True)
historical_features.to_csv(
    output_path,
    index=False,
    date_format="%Y-%m-%d",
)

saved_features = pd.read_csv(
    output_path,
    dtype=identifier_dtypes,
    parse_dates=[
        "purchase_date",
        "previous_received_date",
        "previous_paid_purchase_date",
        "previous_gift_date",
    ],
)
assert len(saved_features) == len(historical_features)
assert list(saved_features.columns) == list(historical_features.columns)

validation_summary = pd.Series({
    "historical_feature_rows": len(historical_features),
    "customers": historical_features["customer_id"].nunique(),
    "products": historical_features["product_id"].nunique(),
    "negative_history_counts": negative_history_count,
    "invalid_previous_dates": invalid_previous_dates,
    "nonzero_first_history_counts": nonzero_first_history_count,
})

print(f"Saved validated historical features to {output_path}")
validation_summary

## Next step

Build historical scoring groups and candidate products. Join these history features to every customer-candidate-product row, then create `label = 1` for products paid for on the target date and `label = 0` for the other candidates. That later table will be saved as `data/processed/catboost_training.csv`.